In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Konfigurasi pengurangan ralat

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



> **Note:** Keluaran beta bagi model pelaksanaan baharu kini tersedia. Model pelaksanaan terarah memberikan lebih fleksibiliti apabila menyesuaikan alur kerja pengurangan ralat anda. Lihat panduan [Model pelaksanaan terarah](/guides/directed-execution-model) untuk maklumat lanjut.


<details>
<summary><b>Versi pakej</b></summary>

Kod dalam halaman ini dibangunkan menggunakan keperluan berikut.
Kami mengesyorkan penggunaan versi ini atau yang lebih baharu.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
Teknik pengurangan ralat membolehkan pengguna mengurangkan ralat Circuit dengan
memodelkan hingar peranti pada masa pelaksanaan. Ini biasanya
menghasilkan overhead pra-pemprosesan kuantum berkaitan latihan model dan
overhead pasca-pemprosesan klasik untuk mengurangkan ralat dalam keputusan mentah
menggunakan model yang dijana.

Primitif Estimator menyokong beberapa teknik pengurangan ralat, termasuk [TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation), [ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne), [PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec), dan [PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options). Lihat [Teknik pengurangan dan penindasan ralat](error-mitigation-and-suppression-techniques) untuk penjelasan setiap satu. Apabila menggunakan primitif, anda boleh menghidupkan atau mematikan kaedah individu. Lihat bahagian [Tetapan ralat tersuai](#advanced-error) untuk butiran.

> **Note:** Sampler tidak menyokong pengurangan ralat, tetapi anda boleh menggunakan pakej [mthree](https://qiskit.github.io/qiskit-addon-mthree/) (pengurangan ralat pengukuran bebas matriks) untuk melakukan pengurangan ralat secara tempatan.

Estimator juga menyokong `resilience_level`. Tahap ketahanan menentukan sejauh mana ketahanan yang dibina terhadap
ralat. Tahap yang lebih tinggi menghasilkan keputusan yang lebih tepat, dengan mengorbankan
masa pemprosesan yang lebih lama. Tahap ketahanan boleh digunakan untuk mengkonfigurasi
pertukaran kos/ketepatan apabila mengaplikasikan pengurangan ralat kepada pertanyaan primitif
anda. Pengurangan ralat mengurangkan ralat (berat sebelah) dalam keputusan dengan memproses
output daripada koleksi, atau ensemble, litar yang berkaitan. Tahap
pengurangan ralat bergantung pada kaedah yang diaplikasikan. Tahap ketahanan
mengabstrakkan pilihan terperinci kaedah pengurangan ralat untuk membolehkan
pengguna menimbang pertukaran kos/ketepatan yang sesuai untuk
aplikasi mereka.

Memandangkan hal ini, setiap tahap sepadan dengan satu atau beberapa kaedah dengan
overhead pensampelan kuantum yang semakin meningkat untuk membolehkan anda bereksperimen
dengan pertukaran masa-ketepatan yang berbeza. Jadual berikut menunjukkan
tahap dan kaedah yang sepadan yang tersedia untuk setiap primitif.

> **Info:** Pengurangan ralat adalah khusus tugasan, jadi teknik yang boleh anda aplikasikan berbeza bergantung pada sama ada anda mensampling taburan atau menjana nilai jangkaan.

<span id="resilience-table"></span>

Estimator menyokong tahap ketahanan berikut. Sampler tidak menyokong tahap ketahanan.

| Tahap Ketahanan | Definisi                                                                                            | Teknik                                                                 |
|------------------|-------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------|
| 0                | Tiada pengurangan                                                                                     | Tiada                                                                      |
| 1 [Lalai]        | Kos pengurangan minimum: Mengurangkan ralat yang berkaitan dengan ralat pembacaan                     | Pentwirlan pengukuran Twirled Readout Error eXtinction (TREX)             |
| 2                | Kos pengurangan sederhana. Biasanya mengurangkan berat sebelah dalam anggaran, tetapi tidak dijamin bebas berat sebelah. | Tahap 1 + Zero Noise Extrapolation (ZNE) dan pentwirlan get

> **Info:** Tahap ketahanan masih dalam beta, jadi overhead pensampelan dan kualiti penyelesaian akan berbeza dari satu Circuit ke Circuit yang lain. Ciri baharu, pilihan lanjutan, dan alat pengurusan akan dikeluarkan secara berterusan. Kaedah pengurangan ralat tertentu tidak dijamin akan diaplikasikan pada setiap tahap ketahanan.

## Konfigurasi Estimator dengan tahap ketahanan
Anda boleh menggunakan tahap ketahanan untuk menentukan teknik pengurangan ralat, atau anda boleh menetapkan teknik tersuai secara individu seperti yang diterangkan dalam [Tetapan ralat tersuai.](#advanced-error)

<details>
<summary>Tahap Ketahanan 0</summary>

Tiada pengurangan ralat yang diaplikasikan kepada program pengguna.

</details>

<details>
<summary>Tahap Ketahanan 1</summary>

Tahap 1 mengaplikasikan **pengurangan ralat pembacaan** dan **pentwirlan pengukuran** dengan mengaplikasikan teknik bebas model yang
dikenali sebagai Twirled Readout Error eXtinction (TREX). Ia mengurangkan ralat pengukuran
dengan mendiagonalkan saluran hingar yang berkaitan dengan pengukuran dengan
membalikkan qubit secara rawak melalui get X sejurus sebelum pengukuran. Sebutan
penskalaan semula daripada saluran hingar pepenjuru dipelajari dengan
menanda aras Circuit rawak yang dimulakan dalam keadaan sifar. Ini membolehkan
perkhidmatan mengeluarkan berat sebelah daripada nilai jangkaan yang terhasil daripada
hingar pembacaan. Pendekatan ini dihuraikan lebih lanjut dalam [Pengurangan ralat pembacaan bebas model untuk nilai jangkaan
kuantum](https://arxiv.org/abs/2012.09738).

</details>

<details>
<summary>Tahap Ketahanan 2</summary>

Tahap 2 mengaplikasikan **teknik pengurangan ralat yang terdapat dalam tahap 1** dan juga mengaplikasikan **pentwirlan get** serta menggunakan **kaedah Zero Noise Extrapolation (ZNE)**.  ZNE mengira
nilai jangkaan yang boleh diperhati untuk faktor hingar yang berbeza
(peringkat amplifikasi) dan kemudian menggunakan nilai jangkaan yang diukur untuk
membuat inferens nilai jangkaan ideal pada had hingar sifar (peringkat ekstrapolasi).
Pendekatan ini cenderung mengurangkan ralat dalam nilai jangkaan, tetapi
tidak dijamin menghasilkan keputusan yang bebas berat sebelah.

![Imej ini menunjukkan graf. Paksi-x berlabel Faktor amplifikasi hingar. Paksi-y berlabel Nilai jangkaan. Garis yang menaik berlabel Nilai yang dikurangkan. Titik-titik berhampiran garis ialah nilai yang diperkuat hingar. Terdapat garis mendatar tepat di atas paksi X yang berlabel Nilai tepat.](../docs/images/guides/configure-error-mitigation/resilience-2.svg "Ilustrasi kaedah ZNE")

Overhead kaedah ini berskala dengan bilangan faktor hingar. Tetapan lalai
mensampling nilai jangkaan pada tiga faktor hingar,
menghasilkan overhead lebih kurang 3x apabila menggunakan tahap ketahanan ini.

Dalam Tahap 2, kaedah TREX membalikkan qubit secara rawak melalui get X sejurus sebelum pengukuran,
dan membalikkan bit yang diukur sepadan jika get X telah diaplikasikan. Pendekatan ini dihuraikan lebih lanjut dalam [Pengurangan ralat pembacaan bebas model untuk nilai jangkaan
kuantum](https://arxiv.org/abs/2012.09738).

</details>

### Contoh
Antara muka `EstimatorV2` membolehkan pengguna bekerja dengan lancar menggunakan pelbagai
kaedah pengurangan ralat untuk mengurangkan ralat dalam nilai jangkaan yang boleh diperhati. Kod berikut menggunakan Zero Noise Extrapolation dan pengurangan ralat pembacaan dengan hanya
menetapkan `resilience_level 2`.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## Tetapan ralat tersuai
Anda boleh menghidupkan dan mematikan kaedah pengurangan dan penindasan ralat individu, termasuk nyahkopling dinamik, pentwirlan get dan pengukuran, pengurangan ralat pengukuran, PEC, dan ZNE. Lihat [Teknik pengurangan dan penindasan ralat](error-mitigation-and-suppression-techniques) untuk penjelasan setiap satu kaedah.

> **Note:** - Tidak semua pilihan tersedia untuk kedua-dua primitif. Lihat [jadual pilihan yang tersedia](runtime-options-overview#options-table) untuk senarai pilihan yang tersedia.
> - Tidak semua kaedah berfungsi bersama pada semua jenis Circuit. Lihat [jadual keserasian ciri](runtime-options-overview#options-compatibility-table) untuk butiran.

<Tabs>
  <TabItem value="Estimator" label="Estimator">
    ```python
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}")
print(f">>> measurement error mitigation is turned on: {estimator.options.resilience.measure_mitigation}")
```
  </TabItem>

  <TabItem value="Sampler" label="Sampler">